# MST-GNN Paper Replication (Google Colab Notebook)

This notebook helps you run the replication code for **MST-GNN** (IEEE TCSS 2024) on **Google Colab (T4 GPU recommended)**.

1. **PyTorch Geometric (PyG) Compatibility**: PyG relies on highly-optimized custom sparse matrix operations (`torch-scatter`, `torch-sparse`). The PyTorch MPS (Metal Performance Shaders) backend on Apple Silicon M1 lacks stable or fully-optimized support for these sparse operations, which can lead to compilation errors or silent fallbacks to slow CPU processing.
2. **Hardware Acceleration**: NVIDIA CUDA (available on Colab T4) is natively supported by PyG with pre-compiled wheels. Training our dynamic graph models (where the graph structure changes daily) runs significantly faster on CUDA GPUs.
3. **Data Fetching**: AKShare fetching can be rate-limited, but running on a cloud instance sometimes behaves differently. However, once fetched, the datasets are cached as pickle files so future runs are instant.

## 1. Set Up Project Workspace

In [1]:
import torch
print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None")


CUDA available: False
GPU: None


In [2]:
import subprocess

# 1. Xem Colab có thực sự nhận GPU không
result = subprocess.run(["nvidia-smi"], capture_output=True, text=True)
print("=== nvidia-smi ===")
print(result.stdout if result.stdout else result.stderr)

# 2. Xem PyTorch version nào đang dùng
import torch
print(f"\nPyTorch version: {torch.__version__}")
print(f"Built with CUDA: {torch.version.cuda}")
print(f"CUDA available: {torch.cuda.is_available()}")


FileNotFoundError: [Errno 2] No such file or directory: 'nvidia-smi'

In [ ]:
!git clone https://github.com/quocnguyen5/mst-gnn-impl.git
%cd mst-gnn-impl

Cloning into 'mst-gnn-impl'...
remote: Enumerating objects: 115, done.
remote: Counting objects: 100% (115/115), done.
remote: Compressing objects: 100% (89/89), done.
remote: Total 115 (delta 36), reused 97 (delta 21), pack-reused 0 (from 0)
Receiving objects: 100% (115/115), 9.18 MiB | 26.63 MiB/s, done.
Resolving deltas: 100% (36/36), done.
/content/mst-gnn-impl


In [7]:
!git pull

remote: Enumerating objects: 27, done.
remote: Counting objects: 100% (27/27), done.
remote: Compressing objects: 100% (12/12), done.
remote: Total 20 (delta 13), reused 14 (delta 8), pack-reused 0 (from 0)
Unpacking objects: 100% (20/20), 20.75 KiB | 758.00 KiB/s, done.
From https://github.com/quocnguyen5/mst-gnn-impl
   1165de0..8c3fa81  main       -> origin/main
Updating 1165de0..8c3fa81
Fast-forward
 MST_GNN_Colab.ipynb     | 1019 ++++++++++++++++++++++++++++++++++++-----------
 data/collector_vn.py    |  456 +++++++++++++++++++++
 data/graph_builder.py   |   30 +-
 experiments/run_main.py |   20 +
 experiments/run_vn.py   |  298 ++++++++++++++
 requirements.txt        |    3 +-
 6 files changed, 1586 insertions(+), 240 deletions(-)
 create mode 100644 data/collector_vn.py
 create mode 100644 experiments/run_vn.py


## 2. Install Dependencies

This cell dynamically installs PyTorch Geometric (PyG) and other packages matching the specific PyTorch and CUDA versions pre-installed in your Colab environment.

In [ ]:
# Install core python packages
!pip install akshare baostock gensim jieba pandas numpy scikit-learn scipy matplotlib seaborn tqdm pyyaml tensorboard

# Install PyTorch Geometric matching the pre-installed PyTorch & CUDA version
import torch
pyt_version = torch.__version__.split('+')[0]
if torch.cuda.is_available():
    cuda_version = torch.version.cuda.replace('.', '')
    print(f"CUDA is available. Version: {torch.version.cuda}")
    !pip install torch-scatter torch-sparse -f https://data.pyg.org/whl/torch-{pyt_version}+cu{cuda_version}.html
else:
    print("Warning: CUDA is NOT available. Running on CPU mode.")
    !pip install torch-scatter torch-sparse -f https://data.pyg.org/whl/torch-{pyt_version}+cpu.html

!pip install torch-geometric

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 21.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.9/47.9 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 63.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.1/10.1 MB 103.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 101.6 MB/s eta 0:00:00
  Created wheel for jsonpath: filename=jsonpath-0.82.2-py3-none-any.whl size=5615 sha256=2eecde97ce82c65396fce9c980606eb79379afee6a03b518b3e2f3cd11d6e9f9
  Stored in directory: /root/.cache/pip/wheels/73/76/e2/980a29341fe37a583ada29594ed529708d5e8e2c0f9d97c3cc
Successfully built jsonpath
Looking in links: https://data.pyg.org/whl/torch-2.11.0+cpu.html
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 681.6/681.6 kB 15.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 49.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.4/64

## 3. Run Experiments

### **A. Run the Main Experiment (CSI 300)**
This script handles:
1. Data collection via AKShare (and caches to raw folders).
2. Technical feature engineering (13 features).
3. Construction of 4 stock network layers (Shareholding, Industry, Topicality/News LDA, Comovement).
4. Splitting chronological snapshots into Train, Val, and Test.
5. Model training (using early stopping).
6. Backtesting and trading simulation.

In [ ]:
#sanity check
!python -m experiments.run_sanity_check


INFO:sanity_check:Starting MST-GNN implementation sanity check...
INFO:sanity_check:Building synthetic datasets...
INFO:sanity_check:Initializing MST-GNN model...
INFO:sanity_check:Starting training loop...
INFO:train:Model parameters: 175,747
INFO:train:Module params: {'encoder': 24576, 'stna': 67072, 'hoff': 75584, 'predictor': 8515}
INFO:train:Device: cpu
2026-06-26 13:06:54.186175: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2026-06-26 13:06:54.273253: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
INFO:sanity_check:Running trainer.train()...
INFO:train:Starting training for 3 epochs...
INFO:train:Train: 5 snapshots, Val: 2 snapshots
INFO:train:Epoch   1/3 (0.5s) | LR: 7.50e

In [8]:
# Cell 2: Kiểm tra nhanh dataset có empty không
import sys
sys.path.insert(0, '/content/mst-gnn-impl')
from data.dataset import DatasetBuilder
import pickle, os

pkl = "data/processed/csi300_snapshots.pkl"
if os.path.exists(pkl):
    with open(pkl, "rb") as f:
        snaps = pickle.load(f)
    print(f"Snapshots: {len(snaps)}")
else:
    print("No cached snapshots found — need to rebuild")


Snapshots: 1084


In [9]:
# Run CSI 300 with mean aggregator (fastest, standard variant)
!python -m experiments.run_main --dataset csi300 --aggregator mean


[Phase 1] Collecting data (loading from cache if available)...
  [Phase 1 done] 0.2s — 284,086 price rows, 5530 industry records

[Phase 2] Preprocessing — computing 13 features & sliding windows...
  [Phase 2 done] 11.4s — 1088 trading days, 282,365 samples
[Phase 3] Building graphs for 1088 trading days (this is the slowest step — ~5-15 min)
  Active stocks in universe: 287
  Building shareholding network (static)...
Empty shareholding data, returning empty network.
  Building industry network (static)...
  Training LDA topic model for topicality network...
Building prefix dict from the default dictionary ...
Loading model from cache /tmp/jieba.cache
Loading model cost 1.204 seconds.
Prefix dict has been built successfully.
  [Graph build]: 100%|█████████████████████| 1088/1088 [21:32<00:00,  1.19s/day]
  [Phase 3 complete] 1088 graphs built in 21.5 min (1.19s/day)

[Phase 4] Creating dataset — matching samples to graphs...
  [Phase 4 done] 125.1s — 1084 snapshots | Train: 758 | Val

In [ ]:
# Live-streaming runner — prints output in real-time instead of waiting until the end
# Use this cell to monitor progress during long runs.
import subprocess, sys, os

proc = subprocess.Popen(
    [sys.executable, "-m", "experiments.run_main",
     "--dataset", "csi300", "--aggregator", "mean"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,   # merge stderr into stdout so everything shows
    text=True,
    bufsize=1,                   # line-buffered
    cwd="/content/mst-gnn-impl",
)

# Print each line as it arrives — no waiting until the end
for line in iter(proc.stdout.readline, ""):
    print(line, end="", flush=True)

proc.wait()
print(f"\nProcess finished with return code: {proc.returncode}")


### **B. Run Ablation Studies**
This compares the full MST-GNN model with several ablation variants:
- Without STNA (no spatial-temporal neighborhood aggregation)
- Without HOFF (replaced with simple concatenation)
- Single-task configurations (movement direction only or ranking only)

In [ ]:
!python -m experiments.run_ablation --dataset csi300

2026-06-26 08:03:24.545296: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
Traceback (most recent call last):
object address  : 0x7a48039467a0
object refcount : 3
object type     : 0xa284e0
object type name: KeyboardInterrupt
object repr     : KeyboardInterrupt()
lost sys.stderr


### **C. Run Network Combination Analysis**
This evaluates the model when using different combinations of the 4 networks (Shareholding, Industry, Topicality, Comovement) to prove that multi-network aggregation improves results.

In [ ]:
!python -m experiments.run_network_analysis --dataset csi300

## 4. Visualize Results

This script plots and displays all saved charts (cumulative return backtest lines, ablation charts, and network combination bar plots).

In [ ]:
from IPython.display import Image, display
import os

results_dir = 'results'
if os.path.exists(results_dir):
    print("Generated figures in 'results/':")
    for file_name in sorted(os.listdir(results_dir)):
        if file_name.endswith('.png'):
            print(f"\nFigure: {file_name}")
            display(Image(filename=os.path.join(results_dir, file_name)))
else:
    print("No results directory found. Please run the experiments first.")

## 5. Extension: Vietnam Stock Market (VN30 / VN100)

This section applies **MST-GNN to the Vietnamese stock market (HOSE exchange)**.
It uses the [`vnstock`](https://github.com/thinh-vu/vnstock) library — free, no API key required, works globally (no IP restrictions).

**Active networks on Vietnam data:**
| Network | Status | Reason |
|---|---|---|
| Comovement | ✅ Active | Computed from OHLCV price correlation |
| Industry | ✅ Active | ICB classification from vnstock |
| Shareholding | ⚪ Empty | Not yet implemented for VN |
| Topicality | ⚪ Empty | Requires Vietnamese NLP (future work) |

> **Note**: Running with 2 active networks instead of 4 produces ~1–2% lower accuracy
> than the full CSI 300 model — still sufficient to validate the approach on a new market.
> This gap is an opportunity for future work (Vietnamese NLP, shareholding data from CafeF).

In [ ]:
# Install vnstock — free Vietnam stock data library (works from any country, no login needed)
!pip install vnstock

### **A. VN30 — Quick Run (30 stocks, fastest)**

VN30 = official index of 30 largest HOSE stocks by market cap and liquidity.
Best for a quick proof-of-concept before committing to the full VN100 run.

- **Estimated time**: ~30–45 min total (data fetch + graph build + 200 epochs)
- **Data period**: 2020-01-02 → 2024-06-30 (~4.5 years, ~1,100 trading days)

In [ ]:
# Run MST-GNN on VN30 with mean aggregator (fastest variant)
# Networks active: Comovement + Industry
# Data is cached in data/raw_vn/ after first run — subsequent runs skip fetching
!python -m experiments.run_vn --universe vn30 --aggregator mean

### **B. VN100 — Full Run (top 100 stocks, recommended for presentation)**

Top 100 liquid/large-cap stocks on HOSE. Richer graph structure → better
comovement and industry networks → more meaningful results.

- **Estimated time**: ~60–90 min on Colab T4 GPU
- **Recommended** for the lab presentation to show real-world scalability

In [ ]:
# Run MST-GNN on VN Top 100 with mean aggregator (default 2020-2024 range)
!python -m experiments.run_vn --universe vn100 --aggregator mean

### **C. Custom Date Range**

Run on a specific sub-period. Useful for:
- **Shorter training**: fewer years = faster iteration during development
- **Period analysis**: compare pre-COVID vs post-COVID market behavior
- **Recency bias check**: test if the model generalizes to recent unseen data

In [ ]:
# Run VN100 on a narrower time window (2021-2024)
# Reduces training time by ~30% while still having enough data for robust results
!python -m experiments.run_vn --universe vn100 --start 2021-01-02 --end 2024-06-30

# Alternatively, run VN30 on the same window for an even faster sanity check:
# !python -m experiments.run_vn --universe vn30 --start 2021-01-02 --end 2024-06-30

### **D. Compare All 3 STNA Aggregator Variants**

Runs MST-GNN with all 3 aggregation strategies and prints a comparison table:

| Variant | Aggregator | Description |
|---|---|---|
| MST-GNN-mean | `mean` | Simple average of neighbor hidden states (fastest) |
| MST-GNN-lstm | `lstm` | LSTM-based aggregation (captures ordering, more expressive) |
| MST-GNN-maxpool | `maxpool` | Element-wise max over neighbors (robust to outliers) |

> Reproduces the aggregator ablation from Table VI of the paper, applied to the Vietnam market.
> **Estimated time: ~3–4 hours on Colab T4.**

In [ ]:
# Run all 3 aggregator variants sequentially on VN100 and print a side-by-side comparison
# Each variant takes ~60-90 min, so total ~3-4 hours
!python -m experiments.run_vn --universe vn100 --aggregator all

### **E. Inspect Cached Results & Display Charts**

In [ ]:
# Check how many temporal graph snapshots were built for each VN universe
# Expected: ~750-1100 snapshots depending on universe size and date range
import pickle, os

print("=== Vietnam Dataset Status ===")
for universe in ["vn30", "vn100"]:
    pkl_path = f"data/processed_vn/{universe}_snapshots.pkl"
    if os.path.exists(pkl_path):
        with open(pkl_path, "rb") as f:
            snaps = pickle.load(f)
        # Estimate train/val/test split (70/10/20)
        n = len(snaps)
        print(f"{universe.upper():6s}: {n} snapshots  "
              f"(train~{int(n*0.7)}, val~{int(n*0.1)}, test~{int(n*0.2)})")
    else:
        print(f"{universe.upper():6s}: not yet built — run experiment first")

In [ ]:
# Display saved backtest charts (cumulative portfolio return vs buy-and-hold benchmark)
from IPython.display import Image, display
import os

print("=== Vietnam Backtest Charts ===")
found = False
for universe in ["vn30", "vn100"]:
    chart_path = f"checkpoints/cumulative_returns_{universe}.png"
    if os.path.exists(chart_path):
        print(f"\n--- {universe.upper()} Cumulative Returns ---")
        display(Image(filename=chart_path))
        found = True

if not found:
    print("No VN charts found yet. Run cells A or B first.")